In [1]:
import anndata as ad
import pandas as pd
import scanpy as sc
from pathlib import Path
import os
import sklearn
import scipy
import numpy as np
import sceleto2 as scl
import pickle
import glob
import os
import scrublet as scr
import matplotlib.pyplot as plt


sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(5,5))
sc.settings.figdir = "./figures"
sc.settings.autosave = False
random_seed = 0

from datetime import date
TODAY = date.today()
print(TODAY)

/home/lsj1022/anaconda3/envs/scenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-13


# 1. Merge Microglia data

In [2]:
paths = [
    '/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/processed/ayhan/ayhan_mg.h5ad', #ayhan
    '/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/processed/galvao/galvao_mg.h5ad', #galvao
    '/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/processed/liu/liu_mg.h5ad', #liu
    '/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/processed/pappalardo_total/pappalardo_total_mg.h5ad', #pappalardo
    '/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/processed/thrupp/thrupp_mg.h5ad', #thrupp
]

In [3]:
studies = [
    'ayhan',
    'galvao',
    'liu',
    'pappalardo',
    'thrupp',
]

# Load
adatas = {
    study: sc.read_h5ad(path)
    for study, path in zip(studies, paths)
}

# Concatenate
adata = ad.concat(
    adatas,
    label='study',
    join='inner',
    merge='same',
    index_unique='-',
)

adata

AnnData object with n_obs × n_vars = 285229 × 14642
    obs: 'fcd_subtype', 'brain_region', 'platform', 'sample_id', 'donor_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'doublet_scores', 'predicted_doublets', 'leiden_05', 'leiden_10', 'celltype_lv1', 'study'
    var: 'mt', 'ribo', 'hb'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'counts'

In [4]:
for key in ['X_pca', 'X_pca_harmony', 'X_umap']:
    adata.obsm.pop(key, None)

In [5]:
adata.X = adata.layers['counts'].copy()

In [6]:
adata.write("/data/SJLEE/Epilepsy_Microglia/notebooks/epilepsy_microglia/data/integrated/snRNAseq_microglia_integrated.h5ad")

## annotate tissue

In [8]:
adata.obs['study'].value_counts()

study
ayhan         130344
liu            93922
thrupp         36088
pappalardo     16602
galvao          8273
Name: count, dtype: int64